<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/8-reranker-instructions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara Reranker Instructions with Qwen3

In this notebook we demonstrate how to use **reranker instructions** with Vectara's `qwen3-reranker`. Reranker instructions let you pass domain-specific guidance to the reranker so it can better score relevance for your particular use case.

We'll cover:
- Baseline query using qwen3-reranker **without** instructions
- **Role-based intent steering**: the same query returns different top-5 results depending on whether the user is a researcher or a developer
- **Steering a saturated baseline**: pulling minority-corpus content into the top-5 when vector retrieval strongly favors the opposite side


## About Vectara

[Vectara](https://vectara.com/) is the Agent Platform for trusted enterprise AI: a unified Agentic RAG platform with built-in multi-modal retrieval, orchestration, and always-on governance. Deploy it on-prem (air-gapped), in your VPC, or as SaaS. Vectara agents deliver grounded answers and safe actions with source citations, step-level audit trails, fine-grained access controls, and real-time policy and factual-consistency enforcement, so teams ship faster with lower risk, and with trusted, production-grade AI agents at scale.

Vectara provides a complete API-first platform for building production RAG and agentic applications:

- **Simple Integration**: RESTful APIs and SDKs (Python, JavaScript) for quick integration into any stack
- **Flexible Deployment**: Choose SaaS, VPC, or on-premises deployment based on your requirements
- **Multi-Modal Support**: Index and search across text, tables, and images from PDFs, documents, and structured data
- **Advanced Retrieval**: Hybrid search combining semantic and keyword matching with state-of-the-art reranking
- **Grounded Generation**: LLM responses with citations and factual consistency scores to reduce hallucinations
- **Enterprise-Ready**: Built-in access controls, audit logging, and compliance (SOC2, HIPAA) from day one

## Getting Started

This notebook assumes you've completed Notebooks 1 and 2:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs) with Boomerang embeddings
- Notebook 2: Ingested AI research papers and Vectara documentation

In [1]:
import os
import requests
import json

# Set up authentication
api_key = os.environ['VECTARA_API_KEY']

# Corpus keys from Notebook 1
research_corpus_key = 'tutorial-ai-research-papers'
docs_corpus_key = 'tutorial-vectara-docs'

# Base URL for Vectara API v2
BASE_URL = "https://api.vectara.io/v2"

# Common headers for all requests
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'x-api-key': api_key
}


def run_query(query_request):
    """Run a Vectara query and print the summary and top results."""
    response = requests.post(f"{BASE_URL}/query", headers=headers, json=query_request)
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

    result = response.json()
    print("\n=== Generated Summary ===")
    print(result['summary'])
    print(f"\n=== Factual Consistency Score: {result.get('factual_consistency_score', 'N/A')} ===")

    print("\n=== Top Search Results ===")
    for i, sr in enumerate(result.get('search_results', [])[:5], 1):
        meta = sr.get('document_metadata', {})
        print(f"\n--- Result {i} (score: {sr.get('score', 'N/A'):.4f}) ---")
        print(f"Document: {sr.get('document_id', 'N/A')}")
        print(f"Title: {meta.get('title', 'N/A')}")
        print(f"Text: {sr['text'][:200]}...")

    return result


print("Setup complete.")

Setup complete.


## Example 1: Baseline Query — Qwen3 Reranker Without Instructions

First, let's run a query using the `qwen3-reranker` **without** any instructions across **both corpora** (research papers and Vectara docs). This establishes a baseline to compare against in the following examples.

Note the API difference from earlier notebooks: instead of `reranker_id`, we use `reranker_name` to select the qwen3 reranker.

In [2]:
QUERY = "How does reranking improve search result quality?"

baseline_request = {
    "query": QUERY,
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            },
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 100
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ]
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

print("=" * 60)
print("BASELINE: qwen3-reranker without instructions")
print("=" * 60)
baseline_result = run_query(baseline_request)

BASELINE: qwen3-reranker without instructions

=== Generated Summary ===
Reranking improves search result quality by refining and reordering the results after the initial retrieval. This process enhances the relevance of the results by using various reranker types, such as neural models, to optimize precision and add diversity or custom business logic. By configuring rerankers, the most relevant and business-critical results are prioritized, ensuring they appear at the top of the search results. This is particularly beneficial in applications requiring advanced neural ranking capabilities, such as multilingual content, where rerankers can significantly enhance relevance scoring and improve result quality [1], [2], [5].

=== Factual Consistency Score: 0.71484375 ===

=== Top Search Results ===

--- Result 1 (score: 0.9991) ---
Document: docs-vectara-com-docs-sdk-python-rerankers
Title: Rerankers
Text: Rerankers enhance the relevance of search results by refining and reordering them
afte

## Example 2: Role-Based Intent Steering

The same question can mean different things to different users. A researcher writing a survey wants peer-reviewed papers with novel techniques; a developer building a production system wants implementation guides and API references. With **reranker instructions**, the same query against the same two corpora returns a top-5 tailored to each intent — no query rewriting, no corpus filtering, nothing changes upstream of the reranker.

We run the same query three ways and compare the top-5:

1. **Baseline** — qwen3-reranker with no instructions (neutral relevance ordering).
2. **Researcher role** — instructions to prefer peer-reviewed academic papers.
3. **Developer role** — instructions to prefer implementation docs and API references.

The query — *"What techniques reduce hallucinations in LLMs?"* — is deliberately chosen to sit in the overlap between the two corpora: the research corpus has academic papers on hallucination mitigation, and the Vectara docs cover grounding and factual-consistency features. That mixed baseline is exactly the situation where instructions have room to steer.


In [3]:
# Query with a mixed baseline: both corpora have relevant content, so the
# reranker has headroom to steer in either direction based on the instructions.
ROLE_QUERY = "What techniques reduce hallucinations in LLMs?"


def role_request(instructions=None):
    """Build a query request, optionally attaching reranker instructions."""
    reranker = {
        "type": "customer_reranker",
        "reranker_name": "qwen3-reranker",
        "limit": 100,
        "cutoff": 0.2,
    }
    if instructions:
        reranker["instructions"] = instructions
    return {
        "query": ROLE_QUERY,
        "search": {
            "corpora": [
                {"corpus_key": research_corpus_key, "lexical_interpolation": 0.005},
                {"corpus_key": docs_corpus_key, "lexical_interpolation": 0.005},
            ],
            "limit": 100,
            "context_configuration": {"sentences_before": 2, "sentences_after": 2},
            "reranker": {
                "type": "chain",
                "rerankers": [reranker, {"type": "mmr", "diversity_bias": 0.05}],
            },
        },
        "generation": {
            "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
            "max_used_search_results": 10,
            "response_language": "eng",
            "enable_factual_consistency_score": True,
        },
    }


RESEARCHER_INSTRUCTIONS = (
    "The user is an academic researcher writing a survey paper. Prioritize "
    "peer-reviewed academic research papers with novel techniques, experiments, "
    "and benchmark results. Deprioritize product documentation and tutorials."
)
DEVELOPER_INSTRUCTIONS = (
    "The user is a software developer building a production search application. "
    "Prioritize practical implementation guides, API reference, configuration "
    "options, and code examples. Deprioritize academic theory and research papers."
)

print("=" * 60)
print("BASELINE — no instructions")
print("=" * 60)
baseline_role_result = run_query(role_request())

print("\n" + "=" * 60)
print("RESEARCHER — prioritize academic papers")
print("=" * 60)
researcher_result = run_query(role_request(RESEARCHER_INSTRUCTIONS))

print("\n" + "=" * 60)
print("DEVELOPER — prioritize implementation docs")
print("=" * 60)
developer_result = run_query(role_request(DEVELOPER_INSTRUCTIONS))


BASELINE — no instructions

=== Generated Summary ===
Techniques to reduce hallucinations in large language models (LLMs) include the use of Hallucination Correctors APIs, which automatically detect and correct factual inaccuracies by comparing generated summaries against trusted source documents. This ensures that the output remains grounded in factual content [2]. Additionally, breaking down summaries into sentences or claims before detection, as seen in methods like RAGAS and TruLens, improves accuracy compared to treating the summary as a whole [5]. Retrieval-augmented generation (RAG) is another approach that enhances trustworthiness by integrating external information to support the generated content [9].

=== Factual Consistency Score: 0.97265625 ===

=== Top Search Results ===

--- Result 1 (score: 0.9745) ---
Document: hallucination-detection-naacl.pdf
Title: Hallucination Detection in RAG Systems
Text: Overall, the table illustrates the multifaceted approaches researchers are

In [4]:
def classify_result(doc_id):
    """Classify a result as 'paper' or 'docs' based on document ID."""
    return 'paper' if doc_id.endswith('.pdf') else 'docs'


print(f"=== Role-Based Intent Steering — top-5 breakdown ===")
print(f"Query: {ROLE_QUERY!r}\n")

for label, result in [("BASELINE         ", baseline_role_result),
                      ("RESEARCHER role  ", researcher_result),
                      ("DEVELOPER role   ", developer_result)]:
    top5 = result.get('search_results', [])[:5]
    papers = sum(1 for sr in top5 if classify_result(sr['document_id']) == 'paper')
    docs = len(top5) - papers
    print(f"{label}  papers/docs = {papers}/{docs}")
    for i, sr in enumerate(top5, 1):
        kind = classify_result(sr['document_id'])
        print(f"  {i}. [{kind:5s}] {sr['document_id']} (score: {sr.get('score', 0):.4f})")
    print()

print("-> Same query, same corpora, same retrieval layer. The instruction")
print("   alone shifts which content rises to the top of the results.")


=== Role-Based Intent Steering — top-5 breakdown ===
Query: 'What techniques reduce hallucinations in LLMs?'

BASELINE           papers/docs = 2/3
  1. [paper] hallucination-detection-naacl.pdf (score: 0.9745)
  2. [docs ] docs-vectara-com-docs-rest-api-correct-hallucinations (score: 0.8866)
  3. [docs ] docs-vectara-com-docs-rest-api-correct-hallucinations (score: 0.8808)
  4. [docs ] docs-vectara-com-docs-rest-api-correct-hallucinations (score: 0.8794)
  5. [paper] hallucination-detection-naacl.pdf (score: 0.8788)

RESEARCHER role    papers/docs = 5/0
  1. [paper] hallucination-detection-naacl.pdf (score: 0.9373)
  2. [paper] hallucination-detection-naacl.pdf (score: 0.8456)
  3. [paper] hallucination-detection-naacl.pdf (score: 0.8393)
  4. [paper] hallucination-detection-naacl.pdf (score: 0.8175)
  5. [paper] hallucination-detection-naacl.pdf (score: 0.7931)

DEVELOPER role     papers/docs = 1/4
  1. [docs ] docs-vectara-com-docs-rest-api-correct-hallucinations (score: 0.8722)
  2.

## Example 3: Steering a Saturated Baseline

The previous example used a query where both corpora surfaced in the baseline top-5 — there was a natural mix to rearrange. A more common production scenario is that vector retrieval is already **saturated** in one direction: all your top results come from one subset of your corpus, and the content your user actually needs is buried below the top-N.

Here we show that reranker instructions can **pull a minority subset of the corpus into the top-5 even when vector retrieval strongly favors the opposite side**. The query `"chunking strategies for long documents"` is an all-docs query by default — Vectara's ingestion docs describe chunking extensively, so they dominate. Academic chunking research exists in the research corpus but never makes it into the top-5 without help.

We reuse the researcher instruction from Example 2 to boost academic papers into the saturated top-5.


In [5]:
# Query where the baseline retrieval is saturated toward Vectara docs.
SATURATED_QUERY = "chunking strategies for long documents"


def saturated_request(instructions=None):
    """Build a query request for the saturated-baseline demo."""
    reranker = {
        "type": "customer_reranker",
        "reranker_name": "qwen3-reranker",
        "limit": 100,
        "cutoff": 0.2,
    }
    if instructions:
        reranker["instructions"] = instructions
    return {
        "query": SATURATED_QUERY,
        "search": {
            "corpora": [
                {"corpus_key": research_corpus_key, "lexical_interpolation": 0.005},
                {"corpus_key": docs_corpus_key, "lexical_interpolation": 0.005},
            ],
            "limit": 100,
            "context_configuration": {"sentences_before": 2, "sentences_after": 2},
            "reranker": {
                "type": "chain",
                "rerankers": [reranker, {"type": "mmr", "diversity_bias": 0.05}],
            },
        },
        "generation": {
            "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
            "max_used_search_results": 10,
            "response_language": "eng",
            "enable_factual_consistency_score": True,
        },
    }


print("=" * 60)
print("BASELINE — no instructions (vector retrieval saturated with docs)")
print("=" * 60)
saturated_baseline_result = run_query(saturated_request())


BASELINE — no instructions (vector retrieval saturated with docs)

=== Generated Summary ===
Chunking strategies for long documents involve dividing the text into manageable parts to optimize retrieval and maintain contextual coherence. Two primary strategies are commonly used:

1. **Sentence-based Chunking**: This strategy creates one chunk per sentence, which is the default method. It provides optimal retrieval accuracy for most datasets by maintaining the semantic integrity of each sentence [2], [3].

2. **Max Characters-based Chunking**: This strategy creates larger chunks up to a specified character limit (max_chars_per_chunk). It balances retrieval speed with contextual coherence, making it suitable for performance-tuned ingestion of larger documents [1], [2].

These strategies allow for flexible document processing, ensuring that the text is divided in a way that supports efficient data retrieval and processing [5].

=== Factual Consistency Score: 0.703125 ===

=== Top Search Re

In [6]:
# Same query, same corpora, same reranker. Only the instruction changes.
print("=" * 60)
print("WITH RESEARCHER INSTRUCTION — academic chunking papers boosted")
print("=" * 60)
saturated_researcher_result = run_query(saturated_request(RESEARCHER_INSTRUCTIONS))


WITH RESEARCHER INSTRUCTION — academic chunking papers boosted

=== Generated Summary ===
Chunking strategies for long documents involve breaking down the document into smaller, manageable parts to facilitate processing and retrieval. One approach is sentence chunking, where each sentence is treated as a separate chunk. This method ensures that each chunk is coherent and contextually relevant. Another strategy is to create larger chunks based on a specified character limit, which balances retrieval speed with maintaining contextual coherence [4]. Additionally, documents can be split into disjoint chunks of a fixed word count, such as 100-word chunks, which can then be indexed for fast retrieval [6]. These strategies help in managing and querying large documents efficiently.

=== Factual Consistency Score: 0.87109375 ===

=== Top Search Results ===

--- Result 1 (score: 0.9520) ---
Document: rag-retrieval-augmented-generation.pdf
Title: Retrieval-Augmented Generation for Knowledge-Inten

In [7]:
print(f"=== Saturated-Baseline Steering — top-5 breakdown ===")
print(f"Query: {SATURATED_QUERY!r}\n")

for label, result in [("BASELINE            ", saturated_baseline_result),
                      ("WITH RESEARCHER     ", saturated_researcher_result)]:
    top5 = result.get('search_results', [])[:5]
    papers = sum(1 for sr in top5 if classify_result(sr['document_id']) == 'paper')
    docs = len(top5) - papers
    print(f"{label}  papers/docs = {papers}/{docs}")
    for i, sr in enumerate(top5, 1):
        kind = classify_result(sr['document_id'])
        print(f"  {i}. [{kind:5s}] {sr['document_id']} (score: {sr.get('score', 0):.4f})")
    print()

# Measure the set churn: how many unique docs entered or left the top-5.
baseline_ids = {sr['document_id'] for sr in saturated_baseline_result.get('search_results', [])[:5]}
researcher_ids = {sr['document_id'] for sr in saturated_researcher_result.get('search_results', [])[:5]}
swapped_in = researcher_ids - baseline_ids
swapped_out = baseline_ids - researcher_ids
print(f"Top-5 churn: {len(swapped_in)} document(s) entered, {len(swapped_out)} left.")
print()
print("-> Without instructions, vector retrieval never surfaces research papers")
print("   for this query — Vectara's chunking docs dominate the top-5. A single")
print("   role instruction is enough to pull academic chunking research into")
print("   the top results even though the query itself is unchanged.")


=== Saturated-Baseline Steering — top-5 breakdown ===
Query: 'chunking strategies for long documents'

BASELINE              papers/docs = 0/5
  1. [docs ] docs-vectara-com-docs-rest-api-create-corpus-document (score: 0.9997)
  2. [docs ] docs-vectara-com-docs-rest-api-create-corpus-document (score: 0.9053)
  3. [docs ] docs-vectara-com-docs-rest-api-create-corpus-document (score: 0.9035)
  4. [docs ] docs-vectara-com-docs-rest-api-create-corpus-document (score: 0.8879)
  5. [docs ] docs-vectara-com-docs-rest-api-create-corpus-document (score: 0.8852)

WITH RESEARCHER       papers/docs = 3/2
  1. [paper] rag-retrieval-augmented-generation.pdf (score: 0.9520)
  2. [paper] rag-retrieval-augmented-generation.pdf (score: 0.8046)
  3. [docs ] docs-vectara-com-docs-build-data-ingestion (score: 0.5197)
  4. [docs ] docs-vectara-com-docs-rest-api-create-corpus-document (score: 0.4904)
  5. [paper] gpt3-language-models.pdf (score: 0.2818)

Top-5 churn: 3 document(s) entered, 0 left.

-> Without